# Laura++ lineshape validation: $f_0(980)$ Flatté and $\rho(770)$ Gounaris–Sakurai

This notebook validates the two new lineshapes in isolated one-resonance models for

\[
D^+ \to \pi^- \pi^+ \pi^+.
\]

Two independent models are used:

1. \(D^+\to[f_0(980)\to\pi^-\pi^+]\pi^+\), with `Flatte.f0_980()`;
2. \(D^+\to[\rho(770)\to\pi^-\pi^+]\pi^+\), with `GounarisSakurai()`.

For each case the notebook shows:

- the **pure lineshape**: real part, imaginary part, \(|R(m)|^2\), and phase;
- the **complete symmetrized amplitude component** on a deterministic `DalitzGrid`;
- the corresponding \(m^2(\pi^+\pi^-)\) projection;
- a **100k-event unweighted toy MC**, generated from an independent weighted phase-space pool.

All amplitude/component/PDF normalization uses only deterministic `DalitzGrid` integration. `PhaseSpaceMC` is used only to generate the candidate pools for the toys.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzGrid,
    DecayChannel,
    DecayModel,
    Flatte,
    GounarisSakurai,
    RealImag,
    Resonance,
    ResonanceContext,
    enable_x64,
    weighted_resample,
)

enable_x64()


## Common channel and plotting helpers


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
m_pi_minus, m_pi_plus, _ = channel.daughter_masses

GRID_N = 700
N_POOL = 1_000_000
N_TOY = 100_000

print("m(D+) =", channel.parent_mass, "GeV")
print("daughter masses =", channel.daughter_masses, "GeV")
print("normalization grid =", GRID_N, "x", GRID_N, "=", GRID_N**2, "points")

def plot_pure_lineshape(masses, values, title, vertical_lines=()):
    m = np.asarray(masses)
    z = np.asarray(values)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

    axes[0, 0].plot(m, z.real)
    axes[0, 0].set_ylabel(r"$\mathrm{Re}\,R(m)$")

    axes[0, 1].plot(m, z.imag)
    axes[0, 1].set_ylabel(r"$\mathrm{Im}\,R(m)$")

    axes[1, 0].plot(m, np.abs(z)**2)
    axes[1, 0].set_ylabel(r"$|R(m)|^2$")

    axes[1, 1].plot(m, np.unwrap(np.angle(z)))
    axes[1, 1].set_ylabel("phase [rad]")

    for ax in axes.flat:
        ax.set_xlabel(r"$m_{\pi\pi}$ [GeV]")
        for x, label in vertical_lines:
            ax.axvline(x, linestyle="--", linewidth=1.0, label=label)
    if vertical_lines:
        axes[0, 0].legend()
    fig.suptitle(title)
    plt.show()

def grid_diagnostics(model, title):
    grid = model.normalization_sample
    intensity = np.asarray(model.intensity(grid.as_dict()))
    weights = np.asarray(grid.weights) * intensity

    fig, ax = plt.subplots(figsize=(7, 6))
    h = ax.hist2d(
        np.asarray(grid.s12),
        np.asarray(grid.s13),
        bins=120,
        weights=weights,
    )
    fig.colorbar(h[3], ax=ax, label=r"grid weight $\times |A|^2$")
    ax.set(
        xlabel=r"$s_{12}$ [GeV$^2$]",
        ylabel=r"$s_{13}$ [GeV$^2$]",
        title=title + " — deterministic model density",
    )
    plt.show()

    s_pm = np.concatenate([np.asarray(grid.s12), np.asarray(grid.s13)])
    w_pm = np.concatenate([weights, weights])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(s_pm, bins=120, weights=w_pm, histtype="step", linewidth=1.6)
    ax.set(
        xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]",
        ylabel="weighted grid intensity",
        title=title + r" — $s_{12}+s_{13}$ projection",
    )
    plt.show()

def make_toy(model, seed_pool, seed_resample, title):
    pool = model.generate_phase_space(N_POOL, seed=seed_pool)
    intensity = model.intensity(pool.as_dict())
    target_weights = pool.weights * intensity

    print(
        title,
        "| finite intensity =",
        bool(jnp.all(jnp.isfinite(intensity))),
        "| finite target weights =",
        bool(jnp.all(jnp.isfinite(target_weights))),
    )

    toy = weighted_resample(
        jax.random.key(seed_resample),
        pool,
        target_weights,
        N_TOY,
        replace=True,
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
    fig.colorbar(h[3], ax=ax, label="events")
    ax.set(
        xlabel=r"$s_{12}$ [GeV$^2$]",
        ylabel=r"$s_{13}$ [GeV$^2$]",
        title=title + f" — {N_TOY:,} toy events",
    )
    plt.show()

    s_pm = np.concatenate([np.asarray(toy.s12), np.asarray(toy.s13)])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(s_pm, bins=120, histtype="step", linewidth=1.6)
    ax.set(
        xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]",
        ylabel="entries / bin",
        title=title + r" toy — $s_{12}+s_{13}$ projection",
    )
    plt.show()

    return toy


# 1. $f_0(980)$ only — Flatté

The model contains only the symmetrized scalar component

\[
D^+\to[f_0(980)\to\pi^-\pi^+]\pi^+.
\]

The Flatté parameters are the Laura++ \(f_0(980)\) preset. The nominal pole mass is set to \(0.965\) GeV, matching the convention used by that preset. The `width` field is set to zero because the Flatté width is supplied by the coupled-channel \(\pi\pi\) and \(K\bar K\) couplings, not by `pole_width`.


In [ ]:
flatte_f0 = Flatte.f0_980()

f0_model = DecayModel(
    channel,
    [
        Resonance(
            "f0_980",
            pair=(0, 1),
            coefficient=RealImag(1.0, 0.0),
            mass=0.965,
            width=0.0,
            spin=0,
            lineshape=flatte_f0,
            resonance_radius=3.0,
            parent_radius=3.0,
        )
    ],
    normalization_resolution=GRID_N,
    normalization_boundary_resolution=20001,
)

print("normalization points =", f0_model.normalization_sample.size)


### 1.1 Pure Flatté lineshape


In [ ]:
m_f0 = jnp.linspace(2.0 * m_pi_plus + 1e-5, 1.20, 2500)

f0_context = ResonanceContext(
    parent_mass=channel.parent_mass,
    daughter_masses=(m_pi_minus, m_pi_plus),
    bachelor_mass=m_pi_plus,
    spin=0,
    pole_mass=0.965,
    pole_width=0.0,
    resonance_radius=3.0,
    parent_radius=3.0,
)

r_f0 = flatte_f0(m_f0, f0_context)

kaon_threshold_charged = 2.0 * flatte_f0.channel2[0][0]
kaon_threshold_neutral = 2.0 * flatte_f0.channel2[1][0]

plot_pure_lineshape(
    m_f0,
    r_f0,
    r"$f_0(980)$ Flatté — pure lineshape",
    vertical_lines=(
        (0.965, r"$m_0$"),
        (kaon_threshold_charged, r"$K^+K^-$ threshold"),
        (kaon_threshold_neutral, r"$K^0\bar K^0$ threshold"),
    ),
)

gamma_pi, gamma_k = flatte_f0.widths(m_f0, f0_context)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(m_f0), np.real(np.asarray(gamma_pi)), label=r"Re $\Gamma_{\pi\pi}$")
axes[0].plot(np.asarray(m_f0), np.real(np.asarray(gamma_k)), label=r"Re $\Gamma_{K\bar K}$")
axes[0].legend()
axes[0].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="real width term [GeV]")

axes[1].plot(np.asarray(m_f0), np.imag(np.asarray(gamma_pi)), label=r"Im $\Gamma_{\pi\pi}$")
axes[1].plot(np.asarray(m_f0), np.imag(np.asarray(gamma_k)), label=r"Im $\Gamma_{K\bar K}$")
axes[1].legend()
axes[1].set(xlabel=r"$m_{\pi\pi}$ [GeV]", ylabel="imaginary width term [GeV]")
plt.show()


### 1.2 Complete symmetrized $f_0(980)$ component on the Dalitz grid


In [ ]:
grid_diagnostics(f0_model, r"$D^+\to f_0(980)\pi^+$")


### 1.3 $f_0(980)$-only toy MC


In [ ]:
toy_f0 = make_toy(
    f0_model,
    seed_pool=3100,
    seed_resample=3101,
    title=r"$D^+\to f_0(980)\pi^+$",
)


# 2. $\rho(770)$ only — Gounaris–Sakurai

The second model contains only

\[
D^+\to[\rho(770)^0\to\pi^-\pi^+]\pi^+.
\]

The lineshape is `GounarisSakurai()`, while the complete amplitude component still includes the same parent/resonance barrier factors, covariant spin-1 angular term, and automatic exchange of the two identical \(\pi^+\).


In [ ]:
gs_rho = GounarisSakurai()

rho_model = DecayModel(
    channel,
    [
        Resonance(
            "rho770",
            pair=(0, 1),
            coefficient=RealImag(1.0, 0.0),
            mass=0.7693,
            width=0.1502,
            spin=1,
            lineshape=gs_rho,
            resonance_radius=3.0,
            parent_radius=3.0,
        )
    ],
    normalization_resolution=GRID_N,
    normalization_boundary_resolution=20001,
)

print("normalization points =", rho_model.normalization_sample.size)


### 2.1 Pure Gounaris–Sakurai lineshape


In [ ]:
m_rho = jnp.linspace(2.0 * m_pi_plus + 1e-5, 1.15, 2500)

rho_context = ResonanceContext(
    parent_mass=channel.parent_mass,
    daughter_masses=(m_pi_minus, m_pi_plus),
    bachelor_mass=m_pi_plus,
    spin=1,
    pole_mass=0.7693,
    pole_width=0.1502,
    resonance_radius=3.0,
    parent_radius=3.0,
)

r_rho = gs_rho(m_rho, rho_context)

plot_pure_lineshape(
    m_rho,
    r_rho,
    r"$\rho(770)$ Gounaris--Sakurai — pure lineshape",
    vertical_lines=((0.7693, r"$m_0$"),),
)

pole_value = complex(np.asarray(gs_rho(jnp.asarray(0.7693), rho_context)))
print("R(m0) =", pole_value)
print("|R(m0)|^2 =", abs(pole_value)**2)
print("phase(m0) [rad] =", np.angle(pole_value))


### 2.2 Complete symmetrized $\rho(770)$ component on the Dalitz grid


In [ ]:
grid_diagnostics(rho_model, r"$D^+\to \rho(770)^0\pi^+$")


### 2.3 $\rho(770)$-only toy MC


In [ ]:
toy_rho = make_toy(
    rho_model,
    seed_pool=3200,
    seed_resample=3201,
    title=r"$D^+\to \rho(770)^0\pi^+$",
)


# 3. Direct visual comparison

This last cell overlays the unlike-sign invariant-mass projections of the two toys. They are **not** normalized to a common physical branching fraction; each model contains one unit-normalized amplitude component with coefficient \(1+0i\). The purpose is only to compare the characteristic scalar Flatté and vector GS shapes after the full three-body kinematics, angular term, symmetrization, and toy generation.


In [ ]:
s_f0 = np.concatenate([np.asarray(toy_f0.s12), np.asarray(toy_f0.s13)])
s_rho = np.concatenate([np.asarray(toy_rho.s12), np.asarray(toy_rho.s13)])

bins = np.linspace(
    min(s_f0.min(), s_rho.min()),
    max(s_f0.max(), s_rho.max()),
    140,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(s_f0, bins=bins, density=True, histtype="step", linewidth=1.7, label=r"$f_0(980)$ Flatté")
ax.hist(s_rho, bins=bins, density=True, histtype="step", linewidth=1.7, label=r"$\rho(770)$ GS")
ax.set(
    xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]",
    ylabel="normalized entries",
    title="Single-resonance toy comparison",
)
ax.legend()
plt.show()
